## 1. Why evaluation exists

A RAG system does two separate jobs: it **finds** information (retrieval) and it **uses** that information to write an answer (generation). Each job can fail on its own, and they can fail in ways that look identical from the outside.

Imagine a friend who has access to a filing cabinet and answers your questions by pulling out documents and summarizing them for you. If they give you a wrong answer, there are several totally different explanations:

- They grabbed the wrong folder from the cabinet (retrieval failure)
- They grabbed the right folder but misread it (generation failure)
- They grabbed the right folder, read it correctly, but added something from their own memory that wasn't in the folder (hallucination)
- They grabbed the right folder, read it correctly, and answered correctly — but answered a slightly different question than the one you asked (relevance failure)

From your seat, all four look the same: "wrong answer." Evaluation exists because **a wrong answer alone tells you nothing about why it's wrong**, and you cannot fix what you can't locate. Evaluation is the practice of separating "wrong" into "wrong because of what" — retrieval, generation, or fit between the two.

## 2. What problems evaluation solves

Three distinct problems, and they're worth keeping separate:

**Problem A: Fluency hides falseness.** An LLM-generated answer is grammatically smooth and confident-sounding whether it's right or wrong. A human reading "Our refund window is 30 days from purchase" has no built-in alarm bell telling them this is fabricated versus pulled from the actual policy doc. Bad answers don't announce themselves. Evaluation exists to catch what your eyes wouldn't catch on their own.

**Problem B: You can't tell if a change helped or hurt.** Say you swap your retrieval method, or change your prompt, or upgrade your embedding model. Did it get better? Worse? For which kinds of questions? Without some systematic way to check, you're just vibing — looking at five examples and guessing. Evaluation gives you a before/after comparison that isn't just anecdote.

**Problem C: Things break quietly over time.** Documents get added, removed, or edited. The underlying model behind generation gets updated by its provider. A system that worked great in January can degrade by March with nobody touching the code, because the data it depends on shifted underneath it. Evaluation is how you notice this before your users do.

## 3. Why humans cannot manually evaluate everything

This sounds obvious — "too many questions, not enough humans" — but it's worth pulling apart *why* that's true, because the reasons are different and each one matters for a different reason.

**Volume.** A production system might answer 50,000 queries a day. Even at 30 seconds per manual review, that's over 400 hours of human labor — per day.

**Continuity.** Evaluation isn't a one-time event. You need it every day, every deploy, every data update. A human team checking by hand can do a one-off audit; they can't be the *ongoing* mechanism, because they have other jobs and need to sleep.

**Consistency.** Suppose two different people review the same answer to "What's the refund window?" One thinks "close enough, it mentioned 30 days" and passes it. The other thinks "it didn't mention the exception for sale items" and fails it. Humans disagree with each other, and even the same human disagrees with their past self on a different day. You need a process that gives the same verdict on the same input — not because humans are bad judges, but because *unrepeatable* judgment can't be used to track whether something is improving.

**Rare but costly failures.** If a failure happens 1 time in 2,000, a human spot-checking 20 examples a day will likely never see it — yet that 1-in-2,000 failure might be the one where the bot tells a customer the wrong legal deadline. Manual review naturally samples the common case, and the common case is rarely the dangerous case.

So the human role doesn't disappear — it moves. Instead of reading every answer, a human defines *what "good" looks like* once, and then a repeatable process applies that definition at scale. (This is the conceptual seed of "automated evaluation," even though I'm deliberately not naming any specific method here.)

## 4. What a production team actually wants to know, per answer

Strip away tooling and ask: if you're responsible for this system, what do you genuinely need to know, every time it answers a question?

Walk through an example. A user asks a support bot: *"Can I get a refund on a sale item after 20 days?"*

The system retrieves a paragraph and generates: *"Yes, all items can be refunded within 30 days of purchase."*

A production team, looking at this single exchange, wants answers to four questions, in this order:

1. **Did we retrieve the right material?**
   Was there a passage in the knowledge base that actually addresses sale-item refund rules? If the real policy document mentions a sale-item exception and the system never pulled that paragraph, the failure started at step one — no amount of good writing afterward can fix a search that grabbed the wrong shelf.

2. **Is the answer actually grounded in what was retrieved, or did it make something up?**
   Suppose the retrieved paragraph says "standard items: 30-day refund" and says nothing about sale items. The generated answer just said "all items," silently erasing the distinction. That's not a retrieval failure — the right text was sitting right there — it's the writing step inventing a confident generalization the source never made.

3. **Does the answer actually address what was asked?**
   The user asked specifically about a *sale item* after *20 days*. An answer about refunds "in general" might be factually lifted straight from a real document and still fail the user, because it never engages with the specific condition (sale item) they asked about.

4. **If something's wrong, where exactly is the break?**
   This is the diagnostic payoff of asking 1–3 separately. "The answer was wrong" is not actionable. "The retriever pulled the standard-refund paragraph instead of the sale-item paragraph" is something an engineer can go fix — maybe the sale-item policy is filed in a document that isn't indexed well, or the search is matching on the word "refund" without picking up "sale item" as a meaningful qualifier.

Notice the shape of this: it's not one question ("is this good?") but a small chain of questions, each isolating one link — *search → faithfulness to source → fit to the actual question*. A production team wants that chain answered for every single response, because a system that's right 95% of the time but can't tell you *which* 5% is failing, or *why*, is much scarier to operate than a system that's right 80% of the time with a clear map of its own weak spots.

That diagnostic chain — find it, ground it, fit it, locate the break — is really the whole discipline of RAG evaluation underneath any specific method you'd later use to measure it.